# 🏭 Ceramics Quality Inspection: Segmentation & Broken-Fragment Counting

Welcome to this project notebook! You will build an end-to-end vision pipeline that automatically **detects and counts broken ceramic fragments** in tile images — a core task in manufacturing quality control.

**The Problem**

A ceramic tile leaves the production line. A camera takes a photo. Your system must answer two questions:
1. *Where* are the broken regions? → **Segmentation** (U-Net)
2. *How many* broken fragments are there? → **Counting** (FNN or CNN counter)

**Pipeline Overview**

1. 🗂️ **Setup** — Clone repo, configure paths for Colab
2. 🔍 **Dataset** — Load image–mask pairs, visualize them
3. 🧱 **U-Net** — Train the segmentation backbone
4. 🔢 **FNN Counter** — Count from a pooled latent vector
5. 🧠 **CNN Counter** — Count from the full spatial latent map
6. ⚖️ **Comparison** — FNN vs CNN: table and bar chart
7. 🚀 **Inference** — Walk through the full pipeline on one image

By the end you will understand why spatial information in the bottleneck matters for counting, and how two architecturally different heads can exploit the same U-Net backbone in different ways.

---
## 1 🗂️ Setup

Clone the project repository and add it to the Python path so that the local modules (`dataset`, `unet`, `counter`, `utils`) can be imported normally.

In [ ]:
# Run this cell once on Colab to clone the repository
!git clone -b week2/warehouse-manager/demo https://github.com/eth-bmai-fs26/project.git
%cd project

In [ ]:
import os
import sys

# Make sure the repo root is on the Python path so local imports work
PROJECT_ROOT = os.path.abspath(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Working directory:", os.getcwd())

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from week2.project1.dataset import SegmentationDataset
from week2.project1.unet import UNet, train_unet
from week2.project1.counter import (
    BrokenCounter,
    BrokenCounterCNN,
    train_counter,
    train_counter_cnn,
)
from week2.project1.utils import count_components

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

---
## 2 🔍 Dataset

We have **6 ceramic tile images**, each paired with a binary segmentation mask that labels the broken regions.

The files follow this naming convention:

| Original image | Segmentation mask |
|---|---|
| `data/original/ceramics-1.jpg` | `data/segmented/ceramics-1-segmented.jpg` |
| `data/original/ceramics-2.jpg` | `data/segmented/ceramics-2-segmented.jpg` |
| … | … |

`SegmentationDataset` handles this via a `_mask_name()` helper that appends `-segmented` to the original stem. Both images are converted to **grayscale** and resized to **256×256**. Mask pixels above 0.5 are set to 1 (broken region), the rest to 0.

In [ ]:
DATA_ROOT = "week2/project1/data"

ds = SegmentationDataset(DATA_ROOT, image_size=(256, 256))
# shuffle=False is important: index i must match broken_labels[i]
dl = DataLoader(ds, batch_size=1, shuffle=False)

print(f"Dataset size : {len(ds)} samples")
print(f"Image shape  : {ds[0][0].shape}   (channels, height, width)")
print(f"Mask shape   : {ds[0][1].shape}   (channels, height, width)")

In [ ]:
# Ground-truth broken fragment counts (order matches sorted dataset)
# ceramics-1: 9, ceramics-2: 5, ceramics-3: 12,
# ceramics-4: 0, ceramics-5: 28, ceramics-6: 12
broken_labels = [9, 5, 12, 0, 28, 12]

fig, axes = plt.subplots(2, len(ds), figsize=(18, 6))
for i in range(len(ds)):
    img, mask = ds[i]
    axes[0, i].imshow(img.squeeze(), cmap="gray")
    axes[0, i].set_title(f"ceramics-{i+1}\n({broken_labels[i]} broken)", fontsize=9)
    axes[0, i].axis("off")
    axes[1, i].imshow(mask.squeeze(), cmap="gray")
    axes[1, i].set_title("GT mask", fontsize=9)
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=11)
axes[1, 0].set_ylabel("Mask", fontsize=11)
plt.suptitle("All image–mask pairs", fontsize=13)
plt.tight_layout()
plt.show()

#### What to look for

- Each mask should show **white blobs** (value 1) over the broken areas and **black background** (value 0) elsewhere.
- The number of distinct white blobs roughly corresponds to the broken count shown in the title — though touching fragments may merge into a single blob.
- Note that ceramics-4 has **0 broken fragments**: its mask should be entirely black.
- ceramics-5 has **28 broken fragments** — expect a dense, fragmented mask.

---
## 3 🧱 U-Net Segmentation Model

The segmentation backbone is a **U-Net** — an encoder-decoder architecture with skip connections. Skip connections route fine-grained spatial detail from the encoder directly to the decoder, allowing the model to produce sharp, pixel-accurate masks even after aggressive downsampling.

**Architecture (`base_ch = 32`, input `256×256`)**

| Stage | Module | Output shape |
|---|---|---|
| Input | — | (B, 1, 256, 256) |
| Encoder block 1 | `ConvBlock(1 → 32)` | (B, 32, 256, 256) |
| Downsample 1 | `Conv2d stride 2` | (B, 32, 128, 128) |
| Encoder block 2 | `ConvBlock(32 → 64)` | (B, 64, 128, 128) |
| Downsample 2 | `Conv2d stride 2` | (B, 64, 64, 64) |
| **Bottleneck** | `ConvBlock(64 → 64)` | **(B, 64, 64, 64)** |
| Upsample 1 | `ConvTranspose2d` + skip from enc2 | (B, 32, 128, 128) |
| Upsample 2 | `ConvTranspose2d` + skip from enc1 | (B, 32, 256, 256) |
| Output | `Conv2d(32 → 1)` | (B, 1, 256, 256) |

The **bottleneck** is the key representation layer. Two methods expose it:
- `get_latent(x)` → applies Global Average Pooling → vector **(B, 64)**
- `get_latent_map(x)` → returns the raw spatial map **(B, 64, 64, 64)**

These two views of the bottleneck feed the FNN and CNN counters respectively.

In [ ]:
unet = UNet(in_ch=1, base_ch=32)
print(unet)
print(f"\nBottleneck channels (latent_dim): {unet.latent_dim}")

In [ ]:
# Train the U-Net.
# With only 6 images the model will overfit — that is expected.
# The goal is a segmentation model good enough to produce
# meaningful bottleneck representations for the counters.
train_unet(unet, dl, epochs=200, lr=1e-3)

#### What to look for

- Loss should steadily decrease from epoch 0 to 180+.
- **Overfitting is intentional here** — with 6 samples, the only goal is to get a segmentation model that faithfully reproduces the training masks, producing informative bottleneck features for the counting heads.

In [ ]:
# Visualize original, ground-truth mask, and U-Net prediction side by side
unet.eval()
unet_device = next(unet.parameters()).device

fig, axes = plt.subplots(3, len(ds), figsize=(18, 9))
for i in range(len(ds)):
    img, mask = ds[i]
    img_t = img.unsqueeze(0).to(unet_device)

    with torch.no_grad():
        pred = torch.sigmoid(unet(img_t)).squeeze().cpu()
    pred_bin = (pred > 0.5).float()

    axes[0, i].imshow(img.squeeze(), cmap="gray")
    axes[0, i].set_title(f"ceramics-{i+1}", fontsize=9)
    axes[0, i].axis("off")

    axes[1, i].imshow(mask.squeeze(), cmap="gray")
    axes[1, i].set_title("GT mask", fontsize=9)
    axes[1, i].axis("off")

    axes[2, i].imshow(pred_bin, cmap="gray")
    axes[2, i].set_title("U-Net pred", fontsize=9)
    axes[2, i].axis("off")

for row, label in enumerate(["Original", "GT mask", "U-Net pred"]):
    axes[row, 0].set_ylabel(label, fontsize=11)

plt.suptitle("U-Net segmentation results", fontsize=13)
plt.tight_layout()
plt.show()

---
## 4 🔢 FNN Counter — Pooled Latent Vector

The first counting approach collapses the bottleneck into a single vector via **Global Average Pooling** and feeds it to a small feedforward network.

```
Bottleneck map (B, 64, 64, 64)
        ↓  Global Average Pooling  [inside UNet.get_latent()]
   Latent vector (B, 64)
        ↓  Linear(64 → 64) + ReLU
        ↓  Linear(64 → 1) + Softplus
   Predicted count λ  (B,)
```

**Why Poisson loss?**
Fragment counts are non-negative integers. The Poisson distribution is the natural model for count data: it models the probability of observing `y` events given a rate `λ`. Minimising `λ − y·log(λ)` encourages the network to predict a rate that matches the observed count. `Softplus` ensures `λ > 0`.

**Architecture summary:**

| Layer | Input shape | Output shape |
|---|---|---|
| `Linear(64 → 64)` + ReLU | (B, 64) | (B, 64) |
| `Linear(64 → 1)` + Softplus | (B, 64) | (B, 1) → squeeze → (B,) |

In [ ]:
counter_fnn = BrokenCounter(latent_dim=unet.latent_dim)
train_counter(unet, counter_fnn, dl, broken_labels, epochs=200)

---
## 5 🧠 CNN Counter — Spatial Latent Map

The second approach skips the pooling step. Instead of a 64-dimensional vector, the counter receives the **full spatial feature map** `(B, 64, 64, 64)` and processes it with convolutional layers before any aggregation.

**Why might spatial structure help for counting?**

Global Average Pooling discards *where* activations are located. For counting, spatial structure is directly informative:
- A dense cluster of activations suggests many small fragments
- The spatial distribution of high-activation regions encodes fragment positions
- Convolutional filters can detect patterns like "many small blobs" that a flat vector cannot easily represent

**Architecture:**

| Layer | Input shape | Output shape | Why |
|---|---|---|---|
| `Conv2d(64→32, 3×3)` + BN + ReLU | (B,64,64,64) | (B,32,64,64) | Detect local patterns in bottleneck |
| `MaxPool2d(2)` | (B,32,64,64) | (B,32,32,32) | Halve spatial size, widen receptive field |
| `Conv2d(32→16, 3×3)` + BN + ReLU | (B,32,32,32) | (B,16,32,32) | Higher-level spatial features |
| `AdaptiveAvgPool2d(1)` (GAP) | (B,16,32,32) | (B,16,1,1) | Aggregate spatially |
| `Flatten` → `Linear(16→1)` → `Softplus` | (B,16) | (B,) | Predict Poisson rate |

The GAP here sits **after** two convolutional layers — the key difference from the FNN, where pooling happens before any learned processing.

In [ ]:
counter_cnn = BrokenCounterCNN(in_channels=unet.latent_dim)
train_counter_cnn(unet, counter_cnn, dl, broken_labels, epochs=200)

---
## 6 ⚖️ Comparison: FNN vs CNN Counter

We now evaluate both counters on all 6 images and compare their predictions against the ground truth. We also include the raw **connected-component count** from the segmentation mask as a purely morphological baseline.

In [ ]:
unet.eval()
counter_fnn.eval()
counter_cnn.eval()

rows = []
for idx in range(len(ds)):
    img, _ = ds[idx]
    img_t = img.unsqueeze(0).to(unet_device)

    with torch.no_grad():
        # Segmentation mask → component count
        logits = unet(img_t)
        seg_mask = (torch.sigmoid(logits) > 0.5).float()
        n_segments = count_components(seg_mask)

        # FNN counter: pooled latent vector
        z = unet.get_latent(img_t)
        fnn_pred = int(torch.round(counter_fnn(z)).item())

        # CNN counter: spatial latent map
        z_map = unet.get_latent_map(img_t)
        cnn_pred = int(torch.round(counter_cnn(z_map)).item())

    rows.append({
        "Image":     f"ceramics-{idx+1}",
        "Actual":    broken_labels[idx],
        "FNN pred":  fnn_pred,
        "CNN pred":  cnn_pred,
        "Segments":  n_segments,
    })

df = pd.DataFrame(rows).set_index("Image")
df

In [ ]:
x = np.arange(len(rows))
width = 0.2

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - 1.5*width, df["Actual"],   width, label="Actual",    color="steelblue")
ax.bar(x - 0.5*width, df["FNN pred"], width, label="FNN pred",  color="tomato")
ax.bar(x + 0.5*width, df["CNN pred"], width, label="CNN pred",  color="seagreen")
ax.bar(x + 1.5*width, df["Segments"], width, label="Segments",  color="goldenrod")

ax.set_xticks(x)
ax.set_xticklabels(df.index, rotation=15)
ax.set_ylabel("Fragment count")
ax.set_title("Broken fragment counting: Actual vs FNN vs CNN vs Morphological segments")
ax.legend()
plt.tight_layout()
plt.show()

#### Interpreting the comparison

| Method | What it sees | Inductive bias |
|---|---|---|
| **Morphological segments** | Binarised prediction mask | Purely geometric: counts connected white blobs |
| **FNN counter** | 64-dim pooled vector | Must encode all count information in a fixed-size summary |
| **CNN counter** | 64×64×64 spatial map | Can learn spatial patterns before aggregating |

With only 6 training samples, both neural counters will likely overfit and predict the correct labels. The important question is: **which architecture generalises better to unseen images?**

The CNN counter is the more principled choice because:
- It preserves spatial structure that directly encodes fragment density and distribution
- Its convolutional layers can detect translation-invariant patterns (e.g. "many small activations in this region")
- The FNN's pooled vector can tell *how much* activation there is, but not *how it is arranged*

---
## 7 🚀 Full Pipeline Inference on a Single Image

Let us walk through the entire pipeline on **ceramics-5** (index 4), which has the highest broken count (28 fragments) — the most challenging sample.

In [ ]:
IDX = 4   # ceramics-5

img, gt_mask = ds[IDX]
img_t = img.unsqueeze(0).to(unet_device)

unet.eval()
counter_fnn.eval()
counter_cnn.eval()

with torch.no_grad():
    # Step 1: segmentation
    logits = unet(img_t)
    seg_pred = (torch.sigmoid(logits) > 0.5).float()
    n_segments = count_components(seg_pred)

    # Step 2: FNN path — pooled latent vector
    z = unet.get_latent(img_t)                     # (1, 64)
    fnn_pred = int(torch.round(counter_fnn(z)).item())

    # Step 3: CNN path — spatial latent map
    z_map = unet.get_latent_map(img_t)             # (1, 64, 64, 64)
    cnn_pred = int(torch.round(counter_cnn(z_map)).item())

# Visualise
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(img.squeeze(), cmap="gray")
axes[0].set_title(f"Original: ceramics-{IDX+1}", fontsize=11)
axes[1].imshow(gt_mask.squeeze(), cmap="gray")
axes[1].set_title("Ground truth mask", fontsize=11)
axes[2].imshow(seg_pred.squeeze().cpu(), cmap="gray")
axes[2].set_title("U-Net prediction", fontsize=11)
for ax in axes:
    ax.axis("off")
plt.suptitle(f"ceramics-{IDX+1} — full pipeline", fontsize=13)
plt.tight_layout()
plt.show()

print("Inference summary")
print("-" * 45)
print(f"Actual broken count              : {broken_labels[IDX]}")
print(f"Morphological segments           : {n_segments}")
print(f"FNN counter prediction           : {fnn_pred}")
print(f"CNN counter prediction           : {cnn_pred}")

#### Takeaways

**Three counting strategies, three philosophies:**

- `count_components` is a **morphological** approach — purely geometric, no learning. It counts distinct white blobs in the binarised mask. It is fast and interpretable, but sensitive to mask quality: touching fragments merge into one blob, and segmentation noise creates spurious extra blobs.

- The **FNN counter** collapses the entire bottleneck into a 64-number summary via Global Average Pooling, then runs a simple two-layer MLP. It learns *how much* signal is in the bottleneck on average, but loses information about *where* activations appear.

- The **CNN counter** processes the bottleneck as a spatial feature map, using two convolutional layers before aggregating. It can detect local patterns (e.g. high-density activation clusters that correspond to many fragments) before collapsing to a single count. This is the architecturally richer approach and is more likely to generalise to unseen images.

> **Key insight:** The choice between FNN and CNN for the counting head is a trade-off between *model simplicity* and *information preservation*. When the input is inherently spatial and the task depends on spatial structure (as counting fragments does), preserving that structure through convolutional processing before aggregation is the better design.